# Итоговый проект: анализ набора соединений

Это шаблон. Скопируйте его и наполните по плану лекции 12: данные из PubChem → свойства RDKit → фильтры → ML → сходство → проверка качества → отчёт.

## 1. Постановка задачи

Опишите: какие вещества, что анализируете, какой результат нужен.

**Задача:** ...

**Набор:** 20 лекарственных веществ

**Результат:** таблица свойств, фильтры, ML, сходство, отчёт с проверками

## 2. Данные из PubChem

Получите SMILES, CID, InChIKey и массу для всех веществ. Сохраните CSV.

In [ ]:
import requests, pandas as pd

compounds = ["aspirin", "ibuprofen", "paracetamol", "caffeine", "ethanol"]
# добавьте остальные 15 веществ из лекции 12

rows = []
for name in compounds:
    url = (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}"
           f"/property/CanonicalSMILES,MolecularFormula,MolecularWeight,InChIKey/JSON")
    try:
        r = requests.get(url)
        p = r.json()["PropertyTable"]["Properties"][0]
        rows.append({"название": name, **p})
    except Exception as e:
        print("Ошибка для", name, e)
df = pd.DataFrame(rows)
df.to_csv("compounds.csv", index=False)
df


## 3. Свойства RDKit

Посчитайте дескрипторы для каждого вещества.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

props = []
for smi in df["CanonicalSMILES"]:
    m = Chem.MolFromSmiles(smi)
    props.append({
        "масса_RDKit": Descriptors.MolWt(m),
        "logP": Descriptors.MolLogP(m),
        "TPSA": Descriptors.TPSA(m),
        "доноры H": Descriptors.NumHDonors(m),
        "акцепторы H": Descriptors.NumHAcceptors(m),
    })
props_df = pd.DataFrame(props)
df = pd.concat([df, props_df], axis=1)
df


## 4. Проверка качества данных

Совпадают ли массы из RDKit и PubChem? Нет ли дубликатов? Все ли SMILES читаются?

In [ ]:
print("Дубликатов:", df["InChIKey"].duplicated().sum())
df["разница масс"] = (df["масса_RDKit"] - df["MolecularWeight"]).abs()
print("Максимальное расхождение масс:", round(df["разница масс"].max(), 3))
df.sort_values("разница масс", ascending=False).head(3)


## 5. Фильтры: Липински и PAINS

In [ ]:
def lipinski_ok(m):
    return (Descriptors.MolWt(m) <= 500 and Descriptors.MolLogP(m) <= 5
            and Descriptors.NumHDonors(m) <= 5 and Descriptors.NumHAcceptors(m) <= 10)

df["Липински"] = [lipinski_ok(Chem.MolFromSmiles(s)) for s in df["CanonicalSMILES"]]
print("Прошли Липински:", df["Липински"].sum(), "из", len(df))


## 6. ML: предсказание «проходит Липински»

Признаки — дескрипторы, цель — колонка Липински. Честно оцените качество: на 20 веществах модель слабая.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X = props_df[['logP', 'TPSA', 'доноры H', 'акцепторы H']]
y = df["Липински"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print("Точность на тесте:", model.score(X_test, y_test))
print("Важность признаков:", dict(zip(X.columns, model.feature_importances_)))


## 7. Сходство

Найдите 3 молекулы, похожие на заданную (например, ибупрофен).

In [ ]:
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity

def fp(smi):
    return AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), radius=2, nBits=1024)

query = fp("CC(C)Cc1ccc(cc1)C(C)C(=O)O")  # ибупрофен
sims = []
for _, row in df.iterrows():
    sims.append(TanimotoSimilarity(query, fp(row["CanonicalSMILES"])))
df["сходство с ибупрофеном"] = sims
df.sort_values("сходство с ибупрофеном", ascending=False)[["название", "сходство с ибупрофеном"]].head(4)


## 8. Графики

Гистограмма масс, распределение logP, диаграмма «прошло/не прошло Липински».

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.hist(df["масса_RDKit"], bins=10)
plt.title("Массы")
plt.subplot(1, 3, 2)
plt.hist(df["logP"], bins=10)
plt.title("logP")
plt.subplot(1, 3, 3)
df["Липински"].value_counts().plot.bar()
plt.title("Липински")
plt.tight_layout()
plt.show()


## 9. Выводы, ограничения и проверка качества

Напишите: что показал анализ, какие ограничения у прогнозов, какие проверки вы выполнили.

**Выводы:**

- ...

**Ограничения:**

- ...

**Проверка качества данных:**

- ...

## 10. Экономика

Сколько заняло бы вручную, сколько — с агентом, стоит ли автоматизировать такой анализ для отдела.

In [ ]:
hours_manual = 12
hours_ai = 2
hour_cost = 600
saved = (hours_manual - hours_ai) * hour_cost
print(f"Экономия за один запуск: {saved:.0f} руб.")
print(f"Экономия за год (1 раз в месяц): {saved * 12:.0f} руб.")
